In [6]:
from typing_extensions import override

#EXTRACT
debugging_mode = True
from pyspark.sql.functions import (
    col, year, month, dayofmonth, weekofyear, dayofweek, when, date_format, monotonically_increasing_id
)

debugging_mode = True
from pyspark.sql import SparkSession
from datetime import datetime, timedelta

spark = SparkSession.builder \
    .appName("S1_01_DIM_DATE") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

# Als er nog geen bron bestaat → maak er één
start_date = datetime(2020, 1, 1)
end_date = datetime(2025, 12, 31)
delta = end_date - start_date
dates = [(start_date + timedelta(days=i),) for i in range(delta.days + 1)]

date_source_df = spark.createDataFrame(dates, ["LogDate"])

# Eerste keer wegschrijven naar Delta (bron simulatie)
date_source_df.write.format("delta").mode("overwrite").save("./data/Log")


In [8]:
#EXTRACT
log_df = spark.read.format("delta").load("./data/Log")

if debugging_mode:
    #DEBUG_CODE
    log_df.show(5)



+-------------------+
|            LogDate|
+-------------------+
|2025-09-02 00:00:00|
|2025-09-03 00:00:00|
|2025-09-04 00:00:00|
|2025-09-05 00:00:00|
|2025-09-06 00:00:00|
+-------------------+
only showing top 5 rows


In [13]:
#TRANSFORM
date_dim = log_df.select(
    col("LogDate").alias("Date"),
    date_format(col("LogDate"), "yyyyMMdd").cast("int").alias("DateSurKey"),  # YYYYMMDD
    dayofmonth(col("LogDate")).alias("Day"),
    weekofyear(col("LogDate")).alias("Week"),
    month(col("LogDate")).alias("Month"),
    year(col("LogDate")).alias("Year"),
    month(col("LogDate")).alias("MonthOfTheYear"),  # Correct: altijd 1-12
    dayofweek(col("LogDate")).alias("DayOfTheWeek"),  # 1=Zondag, 7=Zaterdag
    when(dayofweek(col("LogDate")).between(2,6), True).otherwise(False).alias("IsWeekDay")
).distinct() \
  .withColumn("DateId", monotonically_increasing_id())  # Surrogaat sleutel


In [14]:
if debugging_mode:
    #DEBUG_CODE
    date_dim.show(10)


+-------------------+----------+---+----+-----+----+--------------+------------+---------+------+
|               Date|DateSurKey|Day|Week|Month|Year|MonthOfTheYear|DayOfTheWeek|IsWeekDay|DateId|
+-------------------+----------+---+----+-----+----+--------------+------------+---------+------+
|2025-09-13 00:00:00|  20250913| 13|  37|    9|2025|             9|           7|    false|     0|
|2025-11-04 00:00:00|  20251104|  4|  45|   11|2025|            11|           3|     true|     1|
|2025-12-18 00:00:00|  20251218| 18|  51|   12|2025|            12|           5|     true|     2|
|2025-12-22 00:00:00|  20251222| 22|  52|   12|2025|            12|           2|     true|     3|
|2025-10-06 00:00:00|  20251006|  6|  41|   10|2025|            10|           2|     true|     4|
|2025-11-11 00:00:00|  20251111| 11|  46|   11|2025|            11|           3|     true|     5|
|2025-11-01 00:00:00|  20251101|  1|  44|   11|2025|            11|           7|    false|     6|
|2025-10-19 00:00:00

In [16]:
#LOAD
#LOAD
date_dim.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("./datawarehouse/DateDim")



if debugging_mode:
    #DEBUG_CODE
    print("DateDim succesvol opgeslagen in Delta.")



DateDim succesvol opgeslagen in Delta.
